# AMPR: Adaptive Multimodal Protein Representation

**Thesis:** Dự đoán chức năng protein dựa trên Deep Learning  
**Author:** Nguyen Viet Hung (20224998)  
**Environment:** Google Colab — Python 3.12.13, T4 GPU (15 GB VRAM)

---

## Pipeline tổng quan

```
Phase 1 ── Setup & Clone
Phase 2 ── Download raw data (sequences, GO annotations, PPI)
Phase 3 ── Preprocess: FASTA + GO labels + splits
Phase 4 ── Precompute embeddings (ProteinBERT, ProstT5, BioBERT, Node2Vec)
Phase 5 ── Build GO DAG matrix
Phase 6 ── Create TFRecord files (no contact maps)
Phase 7 ── Train: MF model
Phase 8 ── Train: BP model
Phase 9 ── Train: CC model
Phase 10 ─ Evaluate & Visualize results
```

> **Lưu ý:** Phases 4–6 chỉ cần chạy **một lần**. Lưu kết quả lên Google Drive để tái sử dụng khi session Colab reset.

---
## Phase 0 — Kiểm tra GPU

Đảm bảo runtime đang dùng GPU T4. Nếu không thấy GPU, vào **Runtime → Change runtime type → GPU**.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU không khả dụng — kiểm tra runtime type!')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Phase 1 — Setup: Clone repo & cài thư viện

Clone repo AMPR và cài đặt các dependencies theo `requirements.txt`.  
Tất cả phiên bản được pin cứng để đảm bảo tái lập kết quả.

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ampr.git'  # <-- thay bằng URL repo của bạn
WORK_DIR = '/content/ampr'

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    print(f'Repo đã tồn tại tại {WORK_DIR}')
    !git -C {WORK_DIR} pull

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cài dependencies — mất ~5-10 phút lần đầu
# DGL cần cài riêng với CUDA version phù hợp
!pip install -q torch==2.3.1 transformers==4.41.2 tensorflow==2.16.1
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.3/cu121/repo.html
!pip install -q obonet==1.0.0 biopython==1.84 numpy==1.26.4 scipy==1.14.0
!pip install -q pandas==2.2.0 scikit-learn==1.5.0 pyyaml==6.0.1 tqdm==4.66.4
!pip install -q matplotlib==3.8.4 seaborn==0.13.2

print('\n✓ Tất cả thư viện đã cài xong!')

In [ ]:
# Xác nhận versions
import torch, transformers, numpy, sklearn
print(f'torch        : {torch.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'numpy        : {numpy.__version__}')
print(f'sklearn      : {sklearn.__version__}')

import sys
print(f'Python       : {sys.version}')

---
## Phase 2 — Mount Google Drive (tuỳ chọn nhưng khuyến nghị)

Mount Drive để lưu dữ liệu lớn (embeddings ~2-5 GB) giữa các session.  
Nếu không mount, mọi dữ liệu sẽ mất khi Colab reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục data trên Drive
DATA_DIR = '/content/drive/MyDrive/ampr_data'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f'{DATA_DIR}/embeddings', exist_ok=True)
os.makedirs(f'{DATA_DIR}/tfrecords', exist_ok=True)
os.makedirs(f'{DATA_DIR}/dag_matrices', exist_ok=True)
os.makedirs(f'{DATA_DIR}/checkpoints', exist_ok=True)

# Symlink từ WORK_DIR/data → Drive để code không cần biết về Drive path
if not os.path.exists(f'{WORK_DIR}/data'):
    os.symlink(DATA_DIR, f'{WORK_DIR}/data')

print(f'Data directory: {DATA_DIR}')
print('Drive mounted và symlink tạo xong.')

---
## Phase 3 — Download Raw Data

Download các file cần thiết từ nguồn công khai:

| File | Nguồn | Kích thước | Mục đích |
|---|---|---|---|
| `pdb_seqres.txt.gz` | wwPDB FTP | ~50 MB | Protein sequences |
| `pdb_chain_go.tsv.gz` | EBI SIFTS | ~20 MB | GO annotations |
| `bc-95.out` | RCSB | ~5 MB | Sequence clusters 95% |
| `go-basic.obo` | OBO Library | ~2 MB | GO DAG hierarchy |

> **Thời gian ước tính:** ~10-20 phút (phụ thuộc vào kết nối mạng của Colab)

In [ ]:
os.makedirs('data/raw', exist_ok=True)

downloads = [
    ('ftp://ftp.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt.gz',
     'data/raw/pdb_seqres.txt.gz'),
    ('ftp://ftp.ebi.ac.uk/pub/databases/msd/sifts/flatfiles/tsv/pdb_chain_go.tsv.gz',
     'data/raw/pdb_chain_go.tsv.gz'),
    ('https://cdn.rcsb.org/resources/sequence/clusters/bc-95.out',
     'data/raw/bc-95.out'),
    ('http://purl.obolibrary.org/obo/go/go-basic.obo',
     'data/raw/go-basic.obo'),
]

for url, out_path in downloads:
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
    else:
        print(f'[DOWN] {url}')
        !wget -q "{url}" -O "{out_path}"
        print(f'  → Saved: {out_path}')

print('\n[DONE] Download hoàn tất. Files:')
!ls -lh data/raw/

---
## Phase 4 — Preprocess: Tạo FASTA + GO Labels + Splits

Dùng script `create_nrPDB_GO_annot.py` từ DeepFRI để:
1. Lọc protein có chiều dài 60–1000 aa
2. Loại bỏ protein trùng lặp (90% sequence identity clustering)
3. Propagate GO terms theo DAG hierarchy (parent terms)
4. Tạo train/valid/test splits (90%/10%/5000 proteins cố định)

Output:
- `nrPDB-GO_annot.tsv` — mapping protein → GO terms (MF/BP/CC)
- `nrPDB-GO_sequences.fasta` — sequences
- `nrPDB-GO_train/valid/test.txt` — split files

In [ ]:
# Clone DeepFRI chỉ để lấy preprocessing script
if not os.path.exists('/content/DeepFRI'):
    !git clone -q https://github.com/flatironinstitute/DeepFRI /content/DeepFRI
    print('DeepFRI cloned')
else:
    print('DeepFRI đã có')

In [ ]:
# Cài dependency của DeepFRI preprocessing
!pip install -q obonet networkx

# Chạy preprocessing
ANNOT_PREFIX = 'data/raw/nrPDB-GO'

if os.path.exists(f'{ANNOT_PREFIX}_annot.tsv'):
    print(f'[SKIP] {ANNOT_PREFIX}_annot.tsv đã tồn tại')
else:
    print('[RUN] create_nrPDB_GO_annot.py — mất ~10-15 phút...')
    !python /content/DeepFRI/preprocessing/create_nrPDB_GO_annot.py \
        -sifts data/raw/pdb_chain_go.tsv.gz \
        -bc    data/raw/bc-95.out \
        -seqres data/raw/pdb_seqres.txt.gz \
        -obo   data/raw/go-basic.obo \
        -out   {ANNOT_PREFIX}
    print('[DONE] Preprocessing xong!')

# Xem kết quả
!ls -lh data/raw/nrPDB-GO*
!echo "--- Train/Valid/Test counts ---"
!wc -l data/raw/nrPDB-GO_train.txt data/raw/nrPDB-GO_valid.txt data/raw/nrPDB-GO_test.txt

In [ ]:
# Kiểm tra format của annot.tsv
with open('data/raw/nrPDB-GO_annot.tsv') as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i > 12: break  # chỉ in 13 dòng đầu (headers + 1 data row)

---
## Phase 5 — Precompute Embeddings

Chạy một lần, kết quả lưu vào Drive để tái sử dụng.

| Model | Input | Output | Thời gian |
|---|---|---|---|
| **ProteinBERT** | FASTA sequences | `seq_embeddings.npy` (N, 1024) | ~1-2h |
| **ProstT5** | FASTA sequences | `struct_embeddings.npy` (N, 1024) | ~1-2h |
| **BioBERT** | GO term definitions | `go_emb_{mf,bp,cc}.npy` (C, 768) | ~10 min |

> **Quan trọng:** Tất cả proteins phải được lưu theo cùng một thứ tự (protein_order.json).  
> Thứ tự này được dùng để map split protein IDs → hàng trong .npy files.

In [ ]:
# Bước 5.1: Load sequences từ FASTA và tạo protein_order.json
import json
from Bio import SeqIO

FASTA_PATH = 'data/raw/nrPDB-GO_sequences.fasta'

records = list(SeqIO.parse(FASTA_PATH, 'fasta'))
protein_order = [r.id for r in records]
sequences     = [str(r.seq) for r in records]

print(f'[FASTA] Loaded {len(records)} sequences')
print(f'  Min length : {min(len(s) for s in sequences)}')
print(f'  Max length : {max(len(s) for s in sequences)}')
print(f'  Example ID : {protein_order[0]}')

# Lưu protein order — QUAN TRỌNG để map split IDs → row indices
with open('data/protein_order.json', 'w') as f:
    json.dump(protein_order, f)
print(f'[SAVED] data/protein_order.json ({len(protein_order)} entries)')

In [ ]:
# Bước 5.2: ProteinBERT sequence embeddings
# Output: data/embeddings/seq_embeddings.npy  shape=(N, 1024)

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

SEQ_EMB_PATH = 'data/embeddings/seq_embeddings.npy'

if os.path.exists(SEQ_EMB_PATH):
    print(f'[SKIP] {SEQ_EMB_PATH} đã tồn tại. shape={np.load(SEQ_EMB_PATH).shape}')
else:
    print('[LOAD] Loading ProteinBERT model...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained('Rostlab/prot_bert', do_lower_case=False)
    model = AutoModel.from_pretrained('Rostlab/prot_bert').to(device)
    model.eval()

    BATCH_SIZE = 16  # T4 15GB: giảm nếu OOM
    all_embeddings = []

    print(f'[RUN] Encoding {len(sequences)} sequences (batch={BATCH_SIZE})...')
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc='ProteinBERT'):
            batch_seqs = sequences[i:i+BATCH_SIZE]
            # ProteinBERT cần space giữa các amino acid
            batch_seqs_spaced = [' '.join(s) for s in batch_seqs]

            encoded = tokenizer(
                batch_seqs_spaced,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=1024,
            ).to(device)

            output = model(**encoded)
            # Mean pooling qua sequence length (bỏ [CLS] và [SEP])
            hidden = output.last_hidden_state[:, 1:-1, :]  # (B, L, 1024)
            mask = encoded['attention_mask'][:, 1:-1].unsqueeze(-1).float()
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # (B, 1024)

            all_embeddings.append(emb.cpu().numpy())

    seq_emb = np.concatenate(all_embeddings, axis=0).astype(np.float32)
    print(f'[OUT]  seq_embeddings shape: {seq_emb.shape}')
    np.save(SEQ_EMB_PATH, seq_emb)
    print(f'[SAVED] {SEQ_EMB_PATH} ({seq_emb.nbytes/1e6:.1f} MB)')

    del model  # Giải phóng VRAM
    torch.cuda.empty_cache()

In [ ]:
# Bước 5.3: ProstT5 structural embeddings (từ sequence)
# Output: data/embeddings/struct_embeddings.npy  shape=(N, 1024)

STRUCT_EMB_PATH = 'data/embeddings/struct_embeddings.npy'

if os.path.exists(STRUCT_EMB_PATH):
    print(f'[SKIP] {STRUCT_EMB_PATH} đã tồn tại. shape={np.load(STRUCT_EMB_PATH).shape}')
else:
    print('[LOAD] Loading ProstT5 model...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained('Rostlab/ProstT5', do_lower_case=False)
    model = AutoModel.from_pretrained('Rostlab/ProstT5').to(device)
    model.eval()

    BATCH_SIZE = 8   # ProstT5 lớn hơn, batch nhỏ hơn
    all_embeddings = []

    print(f'[RUN] Encoding {len(sequences)} sequences (batch={BATCH_SIZE})...')
    with torch.no_grad():
        for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc='ProstT5'):
            batch_seqs = sequences[i:i+BATCH_SIZE]
            # ProstT5: prefix '<AA2fold>' để encode từ amino acid sequence
            batch_seqs_spaced = ['<AA2fold> ' + ' '.join(s) for s in batch_seqs]

            encoded = tokenizer(
                batch_seqs_spaced,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=512,
            ).to(device)

            output = model(input_ids=encoded['input_ids'],
                           attention_mask=encoded['attention_mask'],
                           decoder_input_ids=encoded['input_ids'])
            # Dùng encoder hidden states
            hidden = output.encoder_last_hidden_state  # (B, L, 1024)
            mask = encoded['attention_mask'].unsqueeze(-1).float()
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # (B, 1024)

            all_embeddings.append(emb.cpu().numpy())

    struct_emb = np.concatenate(all_embeddings, axis=0).astype(np.float32)
    print(f'[OUT]  struct_embeddings shape: {struct_emb.shape}')
    np.save(STRUCT_EMB_PATH, struct_emb)
    print(f'[SAVED] {STRUCT_EMB_PATH} ({struct_emb.nbytes/1e6:.1f} MB)')

    del model
    torch.cuda.empty_cache()

In [ ]:
# Bước 5.4: PPI embeddings (Node2Vec từ DeepGO DGL graph)
# Output: data/embeddings/ppi_embeddings.npy  shape=(N, 128)
# Proteins không có trong PPI graph → zero vector

PPI_EMB_PATH = 'data/embeddings/ppi_embeddings.npy'

if os.path.exists(PPI_EMB_PATH):
    print(f'[SKIP] {PPI_EMB_PATH} đã tồn tại. shape={np.load(PPI_EMB_PATH).shape}')
else:
    # Download DeepGO PPI data (nếu chưa có)
    if not os.path.exists('data/raw/ppi_mf.bin'):
        print('[INFO] DeepGO PPI .bin files cần download thủ công từ DeepGO repo.')
        print('       Xem: https://github.com/bio-ontology-research-group/deepgont')
        print('       Tạm thời dùng zero embeddings (PPI modality disabled).')

        N = len(sequences)
        ppi_emb = np.zeros((N, 128), dtype=np.float32)
        np.save(PPI_EMB_PATH, ppi_emb)
        print(f'[SAVED] Zero PPI embeddings: {PPI_EMB_PATH} ({ppi_emb.shape})')
    else:
        import dgl
        from node2vec import Node2Vec  # pip install node2vec
        import networkx as nx

        print('[LOAD] Loading PPI DGL graph...')
        graphs, _ = dgl.load_graphs('data/raw/ppi_mf.bin')
        G = graphs[0]

        # Convert DGL → NetworkX cho Node2Vec
        nx_g = G.to_networkx().to_undirected()
        print(f'[PPI]  Nodes: {nx_g.number_of_nodes()}, Edges: {nx_g.number_of_edges()}')

        print('[RUN] Running Node2Vec (128d, 10 walks, length 80)...')
        n2v = Node2Vec(nx_g, dimensions=128, walk_length=80, num_walks=10, workers=2)
        n2v_model = n2v.fit(window=10, min_count=1)

        # Map protein IDs → Node2Vec embeddings (zero nếu không có)
        ppi_emb = np.zeros((len(protein_order), 128), dtype=np.float32)
        found = 0
        for i, pid in enumerate(protein_order):
            node_id = str(G.ndata.get('_ID', {}).get(pid, -1))
            if pid in n2v_model.wv:
                ppi_emb[i] = n2v_model.wv[pid]
                found += 1

        print(f'[PPI]  Found: {found}/{len(protein_order)} proteins in PPI graph')
        print(f'       Zero-vector fallback: {len(protein_order)-found} proteins')
        np.save(PPI_EMB_PATH, ppi_emb)
        print(f'[SAVED] {PPI_EMB_PATH} ({ppi_emb.shape})')

In [ ]:
# Bước 5.5: BioBERT GO term embeddings
# Encode definition text của mỗi GO term → fixed weight matrix cho zero-shot classifier
# Output: data/embeddings/go_emb_{mf,bp,cc}.npy  shape=(C, 768)

import sys
sys.path.insert(0, WORK_DIR)
from scripts.seq2tfrecord import load_GO_annot

import obonet

print('[LOAD] Loading GO annotations and OBO ontology...')
prot2annot, goterms, gonames = load_GO_annot('data/raw/nrPDB-GO_annot.tsv')
go_graph = obonet.read_obo('data/raw/go-basic.obo')

for ont in ['mf', 'bp', 'cc']:
    print(f'\n  {ont.upper()}: {len(goterms[ont])} GO terms')

print('\n[LOAD] Loading BioBERT...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bio_tokenizer = AutoTokenizer.from_pretrained('dmis-lab/biobert-base-cased-v1.2')
bio_model = AutoModel.from_pretrained('dmis-lab/biobert-base-cased-v1.2').to(device)
bio_model.eval()

def encode_go_terms(term_ids, go_graph, tokenizer, model, batch_size=64):
    """Encode GO term definitions via BioBERT [CLS] token."""
    texts = []
    for tid in term_ids:
        if tid in go_graph.nodes:
            node = go_graph.nodes[tid]
            name = node.get('name', tid)
            defn = node.get('def', '').replace('"', '').split('[')[0].strip()
            texts.append(f'{name}. {defn}' if defn else name)
        else:
            texts.append(tid)

    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='BioBERT GO'):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=128).to(device)
            out = model(**enc)
            cls = out.last_hidden_state[:, 0, :]  # [CLS] token (B, 768)
            embeddings.append(cls.cpu().numpy())

    return np.concatenate(embeddings, axis=0).astype(np.float32)

for ont in ['mf', 'bp', 'cc']:
    out_path = f'data/embeddings/go_emb_{ont}.npy'
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
        continue
    print(f'[RUN]  Encoding {len(goterms[ont])} {ont.upper()} GO terms...')
    emb = encode_go_terms(goterms[ont], go_graph, bio_tokenizer, bio_model)
    print(f'[OUT]  go_emb_{ont} shape: {emb.shape}')
    np.save(out_path, emb)
    print(f'[SAVED] {out_path} ({emb.nbytes/1e6:.1f} MB)')

del bio_model
torch.cuda.empty_cache()
print('\n[DONE] Tất cả GO embeddings đã được tạo!')

---
## Phase 6 — Build GO DAG Matrix + Labels

Tạo:
1. **DAG adjacency matrix** `A[i,j]=1` nếu GO term `j` là parent của term `i` — dùng cho DAG loss
2. **Binary label matrices** `labels_{mf,bp,cc}.npy` — mỗi row là 1 protein, mỗi cột là 1 GO term
3. **splits.json** — mapping split name → danh sách protein IDs

In [ ]:
# Bước 6.1: Build DAG adjacency matrices
import networkx as nx

os.makedirs('data/dag_matrices', exist_ok=True)

for ont in ['mf', 'bp', 'cc']:
    out_path = f'data/dag_matrices/dag_matrix_{ont}.npy'
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
        continue

    terms = goterms[ont]
    C = len(terms)
    term2idx = {t: i for i, t in enumerate(terms)}

    dag = np.zeros((C, C), dtype=np.float32)
    edges_added = 0

    for term_id in terms:
        if term_id not in go_graph:
            continue
        child_idx = term2idx[term_id]
        # go_graph là directed: edge từ child → parent
        for parent_id in go_graph.successors(term_id):
            if parent_id in term2idx:
                parent_idx = term2idx[parent_id]
                dag[child_idx, parent_idx] = 1.0
                edges_added += 1

    np.save(out_path, dag)
    print(f'[SAVED] {out_path} — shape={dag.shape}, edges={edges_added}')

print('\n[DONE] DAG matrices xong!')

In [ ]:
# Bước 6.2: Build binary label matrices
prot2idx_global = {pid: i for i, pid in enumerate(protein_order)}
N = len(protein_order)

for ont in ['mf', 'bp', 'cc']:
    out_path = f'data/labels_{ont}.npy'
    if os.path.exists(out_path):
        print(f'[SKIP] {out_path} đã tồn tại')
        continue

    terms = goterms[ont]
    C = len(terms)
    labels = np.zeros((N, C), dtype=np.float32)

    annotated = 0
    for pid, annot in prot2annot.items():
        if pid not in prot2idx_global:
            continue
        row = prot2idx_global[pid]
        labels[row] = annot[ont]
        if annot[ont].sum() > 0:
            annotated += 1

    np.save(out_path, labels)
    pos_rate = labels.sum() / (N * C) * 100
    print(f'[SAVED] labels_{ont}.npy — shape={labels.shape}, '
          f'annotated={annotated}/{N}, positive_rate={pos_rate:.2f}%')

In [ ]:
# Bước 6.3: Tạo splits.json
train_ids = [l.strip() for l in open('data/raw/nrPDB-GO_train.txt')]
valid_ids = [l.strip() for l in open('data/raw/nrPDB-GO_valid.txt')]
test_ids  = [l.strip() for l in open('data/raw/nrPDB-GO_test.txt')]

# Chỉ giữ những ID có trong protein_order
prot_set = set(protein_order)
splits = {
    'train': [p for p in train_ids if p in prot_set],
    'valid': [p for p in valid_ids if p in prot_set],
    'test':  [p for p in test_ids  if p in prot_set],
}

with open('data/splits.json', 'w') as f:
    json.dump(splits, f)

print('[SPLITS]')
for split, ids in splits.items():
    print(f'  {split:5s}: {len(ids):,} proteins')

---
## Phase 7 — Tạo TFRecord files (không có contact map)

Script `seq2tfrecord.py` chuyển đổi FASTA + GO annotations → TFRecord.  
TFRecord được dùng để kiểm tra pipeline và có thể share với các framework khác (TF, JAX).  

> **Lưu ý:** PyTorch training trong Phase 8-10 sẽ đọc trực tiếp từ `.npy` files đã tạo ở Phase 5-6,  
> không phải từ TFRecord. TFRecord hữu ích nếu bạn muốn compare với TensorFlow baseline.

In [ ]:
os.makedirs('data/tfrecords', exist_ok=True)

for split_name in ['train', 'valid', 'test']:
    prefix = f'data/tfrecords/GO_{split_name}'
    first_shard = f'{prefix}_00-of-10.tfrecords'

    if os.path.exists(first_shard):
        print(f'[SKIP] TFRecords cho {split_name} đã tồn tại')
        continue

    print(f'[RUN] Creating TFRecords for {split_name}...')
    !python scripts/seq2tfrecord.py \
        --annot  data/raw/nrPDB-GO_annot.tsv \
        --fasta  data/raw/nrPDB-GO_sequences.fasta \
        --split  data/raw/nrPDB-GO_{split_name}.txt \
        --out_prefix {prefix} \
        --num_shards 10

print('\n[DONE] TFRecords xong:')
!ls -lh data/tfrecords/ | head -20

In [ ]:
# Verify TFRecord: đọc 1 record để kiểm tra
import tensorflow as tf

# Load annotation để biết số GO terms
_, goterms_chk, _ = load_GO_annot('data/raw/nrPDB-GO_annot.tsv')
n_mf = len(goterms_chk['mf'])
n_bp = len(goterms_chk['bp'])
n_cc = len(goterms_chk['cc'])

first_shard = 'data/tfrecords/GO_train_00-of-10.tfrecords'
raw_dataset = tf.data.TFRecordDataset(first_shard)

feature_desc = {
    'prot_id':   tf.io.FixedLenFeature([], tf.string),
    'L':         tf.io.FixedLenFeature([1], tf.int64),
    'seq_1hot':  tf.io.VarLenFeature(tf.float32),
    'mf_labels': tf.io.FixedLenFeature([n_mf], tf.int64),
    'bp_labels': tf.io.FixedLenFeature([n_bp], tf.int64),
    'cc_labels': tf.io.FixedLenFeature([n_cc], tf.int64),
}

for raw_record in raw_dataset.take(1):
    example = tf.io.parse_single_example(raw_record, feature_desc)
    L = int(example['L'].numpy()[0])
    print(f'[VERIFY] prot_id  : {example["prot_id"].numpy().decode()}')
    print(f'         L        : {L}')
    print(f'         seq_1hot : reshape → ({L}, 26) ✓' if len(example['seq_1hot'].values) == L*26 else 'ERROR')
    print(f'         mf_labels: shape=({n_mf},), sum={example["mf_labels"].numpy().sum()}')
    print(f'         bp_labels: shape=({n_bp},), sum={example["bp_labels"].numpy().sum()}')
    print(f'         cc_labels: shape=({n_cc},), sum={example["cc_labels"].numpy().sum()}')
    print(f'         NO ca_dist_matrix key ✓')

print('\n[OK] TFRecord format verified!')

---
## Phase 8 — Training: Molecular Function (MF)

Train AMPR model cho branch MF (~489 GO terms).  
Config: `configs/mf.yaml`

**Training log format:**
```
[EPOCH 01/50] loss=0.3421 (bce=0.2891 dag=0.0530) | α=[0.612, 0.251, 0.137] | val Fmax=0.412
[EPOCH 02/50] loss=0.3187 (bce=0.2701 dag=0.0486) | α=[0.589, 0.278, 0.133] | val Fmax=0.431
```

> **Thời gian ước tính:** 1–2 giờ (50 epochs, T4 GPU)

In [ ]:
# Cập nhật configs để trỏ đúng data paths (absolute paths trên Colab)
import yaml

def update_config_paths(config_path, work_dir):
    """Update relative paths in config to absolute Colab paths."""
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    for key in cfg['data']:
        if isinstance(cfg['data'][key], str) and cfg['data'][key].startswith('data/'):
            cfg['data'][key] = os.path.join(work_dir, cfg['data'][key])

    for key in cfg['output']:
        if isinstance(cfg['output'][key], str):
            cfg['output'][key] = os.path.join(work_dir, cfg['output'][key])
            os.makedirs(os.path.dirname(cfg['output'][key]), exist_ok=True)

    out_path = config_path.replace('.yaml', '_colab.yaml')
    with open(out_path, 'w') as f:
        yaml.dump(cfg, f)
    return out_path

mf_config = update_config_paths(f'{WORK_DIR}/configs/mf.yaml', WORK_DIR)
bp_config = update_config_paths(f'{WORK_DIR}/configs/bp.yaml', WORK_DIR)
cc_config = update_config_paths(f'{WORK_DIR}/configs/cc.yaml', WORK_DIR)

print(f'Colab configs:')
print(f'  MF: {mf_config}')
print(f'  BP: {bp_config}')
print(f'  CC: {cc_config}')

In [ ]:
# Train MF model
print('=' * 60)
print('TRAINING: Molecular Function (MF)')
print('=' * 60)

!python {WORK_DIR}/main.py --config {mf_config} --seed 42

print('\n[DONE] MF training complete!')
!ls -lh {WORK_DIR}/checkpoints/mf/

---
## Phase 9 — Training: Biological Process (BP)

Train AMPR model cho branch BP (~1943 GO terms).  
BP có nhiều GO terms nhất → training chậm hơn MF.  
DAG loss quan trọng hơn ở BP vì hierarchy sâu hơn.

In [ ]:
print('=' * 60)
print('TRAINING: Biological Process (BP)')
print('=' * 60)

!python {WORK_DIR}/main.py --config {bp_config} --seed 42

print('\n[DONE] BP training complete!')
!ls -lh {WORK_DIR}/checkpoints/bp/

---
## Phase 10 — Training: Cellular Component (CC)

Train AMPR model cho branch CC (~320 GO terms).  
CC có ít GO terms nhất → training nhanh nhất.

In [ ]:
print('=' * 60)
print('TRAINING: Cellular Component (CC)')
print('=' * 60)

!python {WORK_DIR}/main.py --config {cc_config} --seed 42

print('\n[DONE] CC training complete!')
!ls -lh {WORK_DIR}/checkpoints/cc/

---
## Phase 11 — Đánh giá & Visualize Kết quả

Parse training logs để hiển thị:
1. Loss curves (BCE vs DAG loss)
2. Alpha weight evolution qua epochs
3. Validation Fmax curves
4. Bảng kết quả cuối cùng (MF / BP / CC)

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def parse_log(log_path):
    """Parse training log → dict of metric lists."""
    data = {'epoch': [], 'loss': [], 'bce': [], 'dag': [],
            'alpha_seq': [], 'alpha_3di': [], 'alpha_ppi': [], 'val_fmax': []}

    epoch_re = re.compile(
        r'EPOCH (\d+)/.*loss=([\d.]+).*bce=([\d.]+).*dag=([\d.]+).*'
        r'α=\[([\d.]+), ([\d.]+), ([\d.]+)\].*val Fmax=([\d.]+)'
    )

    if not os.path.exists(log_path):
        print(f'[WARN] Log not found: {log_path}')
        return data

    with open(log_path) as f:
        for line in f:
            m = epoch_re.search(line)
            if m:
                data['epoch'].append(int(m.group(1)))
                data['loss'].append(float(m.group(2)))
                data['bce'].append(float(m.group(3)))
                data['dag'].append(float(m.group(4)))
                data['alpha_seq'].append(float(m.group(5)))
                data['alpha_3di'].append(float(m.group(6)))
                data['alpha_ppi'].append(float(m.group(7)))
                data['val_fmax'].append(float(m.group(8)))
    return data

branches = ['mf', 'bp', 'cc']
logs = {b: parse_log(f'{WORK_DIR}/logs/{b}_train.log') for b in branches}

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.35)

colors = {'mf': '#2196F3', 'bp': '#4CAF50', 'cc': '#FF9800'}

for row, branch in enumerate(branches):
    d = logs[branch]
    if not d['epoch']:
        continue
    ep = d['epoch']

    # Loss curve
    ax1 = fig.add_subplot(gs[row, 0])
    ax1.plot(ep, d['bce'], label='BCE', color=colors[branch])
    ax1.plot(ep, d['dag'], label='DAG', color=colors[branch], linestyle='--', alpha=0.7)
    ax1.set_title(f'{branch.upper()} — Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    # Alpha weights
    ax2 = fig.add_subplot(gs[row, 1])
    ax2.plot(ep, d['alpha_seq'], label='α_seq', color='#E91E63')
    ax2.plot(ep, d['alpha_3di'], label='α_3di', color='#9C27B0')
    ax2.plot(ep, d['alpha_ppi'], label='α_ppi', color='#FF5722')
    ax2.set_title(f'{branch.upper()} — Alpha Weights')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('α')
    ax2.set_ylim(0, 1); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    # Fmax
    ax3 = fig.add_subplot(gs[row, 2])
    ax3.plot(ep, d['val_fmax'], color=colors[branch], linewidth=2)
    ax3.axhline(max(d['val_fmax']) if d['val_fmax'] else 0,
                color='red', linestyle=':', alpha=0.5, label=f'Best: {max(d["val_fmax"]):.3f}')
    ax3.set_title(f'{branch.upper()} — Val Fmax')
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Fmax')
    ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

fig.suptitle('AMPR Training Results', fontsize=16, fontweight='bold')
plt.savefig(f'{WORK_DIR}/results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/training_curves.png')

In [ ]:
# Bảng kết quả cuối cùng
print('\n' + '='*65)
print(f'{"Branch":>8} | {"Best Fmax":>10} | {"α_seq":>7} | {"α_3di":>7} | {"α_ppi":>7}')
print('-'*65)

for branch in branches:
    d = logs[branch]
    if not d['epoch']:
        print(f'{branch.upper():>8} | {"(no log)":>10}')
        continue

    best_idx = d['val_fmax'].index(max(d['val_fmax']))
    print(
        f'{branch.upper():>8} | '
        f'{d["val_fmax"][best_idx]:>10.3f} | '
        f'{d["alpha_seq"][best_idx]:>7.3f} | '
        f'{d["alpha_3di"][best_idx]:>7.3f} | '
        f'{d["alpha_ppi"][best_idx]:>7.3f}'
    )

print('='*65)
print('α_* = mean alpha weight at best validation epoch')

In [ ]:
# Final alpha comparison bar chart
import matplotlib.pyplot as plt
import numpy as np

branch_labels = ['MF', 'BP', 'CC']
modalities = ['seq (ProteinBERT)', '3Di (ProstT5)', 'PPI (Node2Vec)']
alphas_final = []

for branch in branches:
    d = logs[branch]
    if d['alpha_seq']:
        alphas_final.append([
            d['alpha_seq'][-1],
            d['alpha_3di'][-1],
            d['alpha_ppi'][-1],
        ])
    else:
        alphas_final.append([0.333, 0.333, 0.334])

x = np.arange(len(branch_labels))
width = 0.25
bar_colors = ['#E91E63', '#9C27B0', '#FF5722']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (mod, color) in enumerate(zip(modalities, bar_colors)):
    vals = [alphas_final[j][i] for j in range(len(branch_labels))]
    bars = ax.bar(x + i*width, vals, width, label=mod, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('GO Branch')
ax.set_ylabel('Alpha Weight')
ax.set_title('AMPR Modality Contribution per GO Branch\n(final epoch mean alpha weights)')
ax.set_xticks(x + width)
ax.set_xticklabels(branch_labels)
ax.set_ylim(0, 0.8)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/results/alpha_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/alpha_comparison.png')

---
## Lưu kết quả về Google Drive

Copy checkpoints và results về Drive trước khi session Colab hết hạn.

In [ ]:
import shutil

DRIVE_RESULT_DIR = f'/content/drive/MyDrive/ampr_results'
os.makedirs(DRIVE_RESULT_DIR, exist_ok=True)

for branch in branches:
    src_ckpt = f'{WORK_DIR}/checkpoints/{branch}/best.pt'
    if os.path.exists(src_ckpt):
        dst = f'{DRIVE_RESULT_DIR}/{branch}_best.pt'
        shutil.copy2(src_ckpt, dst)
        print(f'[COPY] {src_ckpt} → {dst}')

for fname in ['training_curves.png', 'alpha_comparison.png']:
    src = f'{WORK_DIR}/results/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_RESULT_DIR}/{fname}')
        print(f'[COPY] {src} → Drive')

print('\n[DONE] Tất cả kết quả đã được lưu về Drive!')

---
## Troubleshooting

| Vấn đề | Giải pháp |
|---|---|
| `CUDA out of memory` | Giảm `batch_size` trong config YAML (256 → 128 → 64) |
| `ModuleNotFoundError: ampr` | Chạy lại ô `os.chdir(WORK_DIR)` ở Phase 1 |
| Session Colab reset | Dữ liệu trên Drive vẫn an toàn; chỉ cần mount lại Drive |
| Loss = 0.0 mọi epoch | Kiểm tra `.npy` files có cùng thứ tự protein với `protein_order.json` không |
| Alpha weights không thay đổi | Kiểm tra learning rate (`lr: 1e-3`) và DAG loss không bị zero |
| Download FTP timeout | Thử lại hoặc dùng alternative mirror |

---

## Tài liệu tham khảo

- **DeepFRI**: Gligorijević et al. (2021) — structure-based GO prediction
- **ProteinBERT**: Brandes et al. (2022) — protein language model
- **ProstT5**: Heinzinger et al. (2023) — structure-aware protein LM
- **CAFA evaluation**: Zhou et al. (2019) — Fmax, Smin metrics
- **True Path Rule**: GO Consortium — DAG hierarchy constraint